# Course notebook — Foundations of Intelligence

Understand intelligence as search, learning, representation, decision-making, and feedback before studying language models.

Use Python 3.10 or later. All inputs are included and the workbook runs without API keys. Each lecture demonstration is a small model of the lecture mechanism. Run it, compare the result with your prediction, and use the section checkpoint to record what you learned.


## Practice case · The alarms at 2 a.m.

Priya is the overnight dispatcher for a regional transit agency. Last Friday, two reports about an elevator outage went into a low-priority queue. A separate report sounded urgent because it was written in all caps. The operations lead wants a smarter dispatcher. Priya wants disabled riders to reach a working station and wants the on-call engineer to receive fewer false alarms.

The workplace scenario is fictional.


## Goals and representations


### Lecture 01 · What Intelligence Means

Treat intelligence as a collection of capabilities rather than a mystical substance. Separate perception, representation, memory, learning, reasoning, planning, action, and adaptation.

#### Two queues, two kinds of success

Priya receives ‘Lift failed again, at North Gate.’ The same wording can mean a functioning lift has a recurring fault or that a passenger is currently stuck. Which decision is the system supposed to help her make?

Split the task into identifying the asset, estimating the situation, and deciding who owns the response. A summary can be accurate while the dispatch outcome is wrong. Measure time to the right responder and count missed urgent cases before judging the system by how well it writes.

#### Read the mechanism

**goal-directed behavior**

Goal-directed behavior means selecting actions to reduce distance to an explicit objective rather than producing unconstrained text. Engineers care because fluent language can mask the absence of a measurable goal.

An incident router should minimize misroutes and escalation time, not maximize eloquent ticket summaries.

**rational agents**

Rational agents choose actions that maximize expected utility toward a goal given perceived state and known constraints. The design question is whether the system's action policy aligns with business utility, not model confidence.

A lending assistant should prefer declining uncertain high-risk cases when false approvals cost more than false declines.

**bounded rationality**

Bounded rationality acknowledges limited compute, time, memory, and information—systems must satisfice within budgets. Production AI rarely has the luxury of exhaustive search or perfect retrieval.

An on-call copilot stops after three retrieval attempts within a 5-second latency SLO instead of searching until theoretical certainty.

**capability decomposition**

Capability decomposition splits intelligence into perception, memory, learning, planning, and action so teams can own, test, and debug each part. Without it, fluent outputs hide which capability failed.

Incident routing can fail in classification while generation still reads naturally—decomposition exposes the failing box.

**feedback**

Feedback closes the loop: outcomes from actions update beliefs, models, or policies for subsequent decisions. Without feedback channels, the same mistakes repeat indefinitely.

Misrouted tickets returned by engineers should update routing features so the error rate on that category is trackable week over week.

#### Worked example

**Situation:** A support team routes incidents without mistaking fluent descriptions for reliable decisions. New hires paste long customer narratives into a shared inbox and guess severity from tone.

**Baseline:** A keyword-free queue that assigns tickets round-robin regardless of content—fast but blind to outage language.

**Application:** Decompose incident handling into perception (parse subject/body), representation (severity features), memory (recent duplicates), and decision (route to on-call vs backlog). Map each capability to an observable checkpoint before any model is introduced.

**Test cases:** (1) Normal: "API latency elevated in us-east" → P2 routing. (2) Boundary: empty body with P1 in subject only. (3) Adversarial: polite prose hiding "data loss" and "all regions down."

**Measurement:** Track precision@P1, median time-to-on-call, and false-P1 rate per 100 tickets; compare capability-map pipeline vs round-robin.

**Design question:** Which capability—perception, representation, or decision—would fail first if you removed human review, and what evidence from the three cases proves it?

**Predict before running.** Intelligence is system behavior under goals and constraints, not a property inferred from fluent language. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 1.1: What Intelligence Means"""

GOAL = "route P1 incidents to on-call"
CAPABILITIES = ["perceive", "represent", "decide", "act"]
ticket = "All regions down — writes failing"
features = set(ticket.lower().split())
severity = "P1" if {"down", "failing"} & features else "P2"
trace = {cap: cap for cap in CAPABILITIES}
trace["represent"] = sorted(features)
trace["decide"] = severity
print({"goal": GOAL, "trace": trace})


#### Interpret and apply

Create a capability map for a familiar human task.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. If the team removed explicit goal-directed behavior from the router, what would still look intelligent but fail on the book scenario?
2. How would you distinguish a failure in bounded rationality from a failure in capability decomposition using ticket logs?
3. What is the simplest baseline that proves fluent language alone is insufficient for routing?

<details><summary>Check your reasoning</summary>

Q1: Without goals, the system might still produce coherent summaries while routing P1 outages to low-priority queues—observable as high false-negative rate on outage keywords. Q2: Bounded-rationality failures show resource limits (timeouts, truncated context); decomposition failures show the wrong module owning the decision. Q3: Round-robin assignment with the same evaluation cases; it ignores content and fails adversarial polite-outage cases.

</details>


### Lecture 02 · From Symbols to Statistics

Trace the path from symbolic rules and expert systems to statistical learning, deep learning, foundation models, and agents. Each era solved different problems and retained useful ideas.

#### Keep the rule the dispatcher can defend

A sensor reports that a passenger is trapped. A second report mentions a lift that has been ‘temperamental’. Priya can explain an immediate dispatch for the first case. The second case has many historical variations.

A hard rule is appropriate for the verified emergency because the consequence is clear. A learned classifier can rank ambiguous language for review. Keeping those decisions in separate layers also makes it possible to tell whether a bad outcome came from the policy or the estimate.

#### Read the mechanism

**symbolic AI**

Symbolic AI represents knowledge as explicit rules, facts, and logical relations rather than learned weights. It remains valuable when constraints are crisp, auditable, and change infrequently.

A tax-credit eligibility checker can encode statutory thresholds as rules that always produce the same answer for the same inputs.

**expert systems**

Expert systems capture domain heuristics in if-then rules curated by specialists, often with explanation traces. They trade flexibility for transparency and predictable behavior in narrow domains.

A manufacturing diagnostic system asks sequential sensor questions and explains which rule fired when recommending a shutdown.

**knowledge representation**

Knowledge representation chooses how facts, relations, and uncertainty are stored—graphs, frames, schemas, or vectors. The representation determines what queries and updates are cheap or hard.

Modeling product compatibility as a graph makes 'works-with' queries fast; flattening to text loses compositional structure.

**statistical learning**

Statistical learning infers patterns from data with explicit assumptions about noise, independence, and generalization. It replaced brittle hand rules where variability and scale made manual encoding impractical.

Spam filtering learned from labeled inboxes outperforms keyword lists when attackers vary phrasing continuously.

**deep learning**

Deep learning stacks differentiable layers that learn hierarchical features from raw inputs. It excels when hand-crafted features are incomplete but demands data, compute, and careful evaluation.

Vision models learn edge and shape detectors automatically where manual feature design for every object class is infeasible.

#### Worked example

**Situation:** The same support team tried a brittle rule engine: IF body CONTAINS "urgent" THEN P1. Marketing emails now preempt real outages.

**Baseline:** Hand-maintained regular expressions over ticket text with no learning loop.

**Application:** Run the rule engine on historical tickets, log brittleness points, then overlay frequency statistics on token co-occurrence with confirmed P1 labels—showing where symbols help (exact SKU codes) and where statistics help (paraphrased outages).

**Test cases:** (1) Normal: exact match "URGENT: payment gateway offline." (2) Boundary: "urgent feature request" vs "urgent—revenue stop." (3) Adversarial: attacker marks newsletter as URGENT.

**Measurement:** Compare rule-only precision/recall against a bag-of-words logistic baseline on a frozen 500-ticket set; tabulate false P1 cost.

**Design question:** For which incident classes should you keep symbolic rules in production even after adding statistical models?

**Predict before running.** New paradigms usually absorb rather than erase earlier engineering techniques. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 1.2: From Symbols to Statistics"""

RULES = [("urgent", "P1"), ("question", "P3")]
tickets = [
    "URGENT: payment gateway offline",
    "urgent feature request for dashboard",
    "question about invoice format",
]
def rule_route(text):
    lower = text.lower()
    for kw, sev in RULES:
        if kw in lower:
            return sev
    return "P2"
for t in tickets:
    print({"ticket": t[:40], "route": rule_route(t)})


#### Interpret and apply

Implement a tiny rule engine and document where it becomes brittle.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Where did symbolic AI help in the incident router, and where did it become brittle?
2. What symptom would indicate the team jumped to deep learning too early?
3. What simpler statistical baseline should precede any neural approach?

<details><summary>Check your reasoning</summary>

Q1: Symbols help for exact policy codes; brittleness appears when paraphrases bypass keywords. Q2: Small labeled set still dominated by regex false positives yet team deploys a large model without slice metrics. Q3: Logistic regression on TF–IDF with the same 500-ticket eval.

</details>


**Section checkpoint.** Write a support-routing goal and identify an example where a fixed keyword rule misroutes a request.


## Search and mathematics


### Lecture 03 · Search, Planning, and Decisions

See classical search as an explicit form of reasoning. Learn how state, actions, transition models, costs, heuristics, and stopping rules turn a vague goal into an algorithm.

#### Draw the station before choosing a route

The shortest walking route to a working lift uses a gate locked after midnight. A noticeably longer route stays accessible. Priya must update the rider before asking maintenance for a repair estimate.

Represent gates and locks as constraints in the state graph. If a route search receives only distance, it can return an unusable route with a perfect distance score. Test the route on a changed lock state before comparing the search algorithms.

#### Read the mechanism

**state spaces**

A state space enumerates all configurations a system can occupy plus the actions that move between them. Explicit state models make search, planning, and verification tractable.

Warehouse robots represent position and load status as state; illegal moves (overweight pickup) are edges you never traverse.

**breadth-first search**

Breadth-first search expands nodes level by level, guaranteeing shortest path in unweighted graphs. It is the baseline for optimal reachability before adding heuristics.

In a grid maze, BFS finds the minimum-step route from start to exit by exploring all distance-1 cells before distance-2.

**A***

A* expands the lowest estimated total-cost node first, combining path cost g(n) with heuristic h(n) toward the goal. With an admissible heuristic it finds optimal paths while often expanding fewer nodes than BFS.

In a grid maze, A* with Manhattan distance typically expands fewer cells than BFS while returning the same shortest path.

**heuristics**

Heuristics estimate remaining cost or promise of partial solutions to guide search toward promising branches. Good heuristics cut compute; bad ones waste it or break optimality guarantees.

Manhattan distance guides grid navigation; an overestimated heuristic can make A* suboptimal or incomplete.

**planning**

Planning sequences actions to reach a goal given a model of state transitions, costs, and constraints. It separates deliberation from execution so plans can be validated before side effects occur.

A deployment planner orders database migration before code rollout because the transition model forbids incompatible schema states.

#### Worked example

**Situation:** Overnight, three dependent services fail in sequence; the router must find a valid escalation path under on-call availability constraints.

**Baseline:** Greedy "pick highest severity keyword" with no search over dependency order.

**Application:** Model on-call slots as a graph: states are (open_incidents, assigned_engineer), actions are assign/defer/escalate, costs are SLA minutes. Run BFS for shortest escalation chain, then A* with heuristic = estimated SLA breach time.

**Test cases:** (1) Normal: single P1 with one qualified on-call. (2) Boundary: P1 when primary on-call is busy but secondary exists. (3) Adversarial: circular dependency declarations causing infinite defer loops.

**Measurement:** Path length, expanded nodes (BFS vs A*), and SLA minutes saved vs greedy on synthetic graphs.

**Design question:** What stopping rule prevents the search from exploring defer loops while still finding a valid escalation?

**Predict before running.** Reasoning can be viewed as controlled search over possible states or candidate solutions. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 1.3: Search, Planning, and Decisions"""

GRAPH = {"start": ["oncall_a", "oncall_b"], "oncall_a": ["lead"], "oncall_b": ["lead"], "lead": []}
GOAL = "lead"
def bfs(start, goal):
    queue = [(start, [start])]
    seen = {start}
    while queue:
        node, path = queue.pop(0)
        if node == goal:
            return path
        for nxt in GRAPH.get(node, []):
            if nxt not in seen:
                seen.add(nxt)
                queue.append((nxt, path + [nxt]))
    return None
print("escalation path:", bfs("start", GOAL))


#### Interpret and apply

Implement breadth-first search and A* on the same maze.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why is explicit state representation necessary for incident escalation?
2. How would you detect a bad heuristic in A* for routing?
3. What non-search baseline should you beat before claiming planning adds value?

<details><summary>Check your reasoning</summary>

Q1: Without state you cannot represent busy on-call or deferred tickets—greedy picks repeat failed assignments. Q2: A* expands as many nodes as BFS or returns suboptimal paths. Q3: Fixed playbook order regardless of availability graph.

</details>


### Lecture 04 · The Mathematics Engineers Need

Develop intuition for vectors, matrices, probability, distributions, statistics, entropy, gradients, and optimization without turning the book into a mathematics degree.

#### An average that hides thirty minutes

Five dispatches take 3, 4, 4, 5, and 30 minutes. The manager asks whether the new routing rule is faster. Priya remembers the passenger attached to the last number.

The median is 4 minutes and the mean is 9.2. They answer different questions. Compare typical speed with the fraction over a service deadline so a system can improve its average while still exposing an unacceptable tail.

##### Turn the numbers into a decision

| Routing plan | Observed minutes | Median | Cases taking over 10 minutes |
|---|---|---:|---:|
| Current | 3, 4, 4, 5, 30 | 4 | 1 of 5 |
| Review-first trial | 4, 5, 5, 6, 8 | 5 | 0 of 5 |

The current plan serves a typical case a minute faster. The review-first trial avoids the observed 30-minute wait and has a lower mean, 5.6 minutes versus 9.2. Which matters more depends on the dispatch policy. If a rider must be contacted within 10 minutes, the deadline is directly relevant. If there is only one dispatcher, the extra minute on everyday work can still matter.

Five cases are enough to see the trade-off, and far too few to estimate the worst week. Priya would want to record more overnight cases and compare repeat incidents at the same station before endorsing a threshold. The numbers should make her decision more accountable rather than replacing her judgment.

#### Read the mechanism

**vectors**

Vectors represent objects as numeric arrays so similarity, direction, and composition become computable. They underpin embeddings, attention, and most modern ML pipelines.

Representing users and items as vectors lets recommendation score candidates with a dot product in milliseconds.

**matrix transformations**

Matrix transformations apply linear maps that rotate, scale, or project vector spaces—core to neural layers and attention projections. Understanding them clarifies why depth composes operations.

An embedding layer is a matrix multiply that maps one-hot token indices into dense vectors.

**probability**

Probability quantifies uncertainty over outcomes, enabling expectations, risk calculations, and principled decisions under incomplete information. ML outputs are almost always distributions, not certainties.

A fraud scorer outputs P(fraud); finance uses that probability with loss asymmetries, not a raw boolean.

**entropy**

Entropy measures uncertainty or information content in a distribution—high when outcomes are evenly spread, low when one dominates. It guides feature selection, decision trees, and regularization.

A classifier with 95% softmax mass on one class is low-entropy and cheap to trust for routing; a flat distribution signals ambiguity worth escalating.

**gradient descent**

Gradient descent adjusts parameters in the direction that most reduces loss, using gradients computed from training examples. It is the workhorse optimizer behind most neural network training.

One SGD step on linear regression moves weights toward the line minimizing squared error on the mini-batch.

#### Worked example

**Situation:** Engineers want to cluster incident descriptions by semantic similarity to detect duplicate outages flooding the inbox.

**Baseline:** Jaccard similarity over raw word sets without normalization.

**Application:** L2-normalize TF vectors, compute cosine similarity, apply softmax over candidate duplicates, and walk one gradient step on a tiny linear scorer trained to predict human duplicate labels.

**Test cases:** (1) Normal: "DB replica lag" vs "database replication delay." (2) Boundary: identical tokens, different negation ("not a duplicate"). (3) Adversarial: padded boilerplate text inflating dot products.

**Measurement:** Duplicate-detection F1, cosine distribution histogram, and calibration of similarity thresholds.

**Design question:** When does cosine similarity on unnormalized vectors systematically rank the wrong ticket pair highest?

**Predict before running.** Mathematics is a compact language for relationships, uncertainty, and change. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 1.4: The Mathematics Engineers Need"""

import math
a = [3.0, 0.0, 1.0]
b = [2.0, 0.0, 2.0]
def cosine(u, v):
    dot = sum(x*y for x, y in zip(u, v))
    nu = math.sqrt(sum(x*x for x in u))
    nv = math.sqrt(sum(y*y for y in v))
    return dot / (nu * nv)
def softmax(xs):
    m = max(xs)
    ex = [math.exp(x - m) for x in xs]
    s = sum(ex)
    return [e/s for e in ex]
sims = [cosine(a, b), cosine(a, a), cosine(b, b)]
print("cosines:", [round(c, 3) for c in sims])
print("softmax:", [round(p, 3) for p in softmax(sims)])


#### Interpret and apply

Compute dot products, cosine similarity, softmax, and one gradient update by hand.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why normalize vectors before comparing incident embeddings?
2. What failure looks like when softmax is applied to unscaled logits in duplicate detection?
3. What baseline similarity should cosine beat on the duplicate task?

<details><summary>Check your reasoning</summary>

Q1: Unnormalized vectors overweight frequent boilerplate tokens. Q2: One pair dominates probability mass despite moderate cosine. Q3: Jaccard or raw dot product without normalization.

</details>


**Section checkpoint.** Trace BFS and A* on the same grid. Record both path length and states expanded.


## Learning and decisions


### Lecture 05 · Learning and Generalization

Distinguish memorization from generalization and training from inference. Understand data-generating processes, inductive bias, overfitting, underfitting, and distribution shift.

#### A new station breaks the naming shortcut

The training set calls every lift near the old North Gate ‘NG’. A new station uses the same abbreviation for a bus stand. Early pilot accuracy looked excellent on a random sample of old tickets.

Hold out whole stations or newer periods. A random split can put near-duplicate messages in training and evaluation. Comparing results on an unseen station tests whether the routing behavior depends on a general signal or on a remembered abbreviation.

#### Read the mechanism

**training**

Training fits model parameters to data by minimizing a loss over many examples. It defines what behavior the model is rewarded for and must be separated from inference in operations.

Fine-tuning a classifier on support tickets teaches phrasing patterns that inference-time prompts alone may not stabilize.

**inference**

Inference applies a trained model to new inputs to produce predictions or generations. Serving latency, cost, and correctness are measured here—not during training.

A production chatbot runs inference on every user message; batching ten requests changes throughput but not the trained weights.

**generalization**

Generalization is performance on unseen data drawn from the deployment distribution, not memorization of training examples. The central engineering question is whether the system will work next month on real users.

A memorizing model hits 100% on training tickets but fails on new product names never seen during training.

**bias and variance**

Bias is systematic underfitting from overly simple models; variance is sensitivity to training noise from overly complex ones. Tuning trades these errors against compute and data volume.

A linear model underfits nonlinear fraud patterns (high bias); a huge tree overfits small samples (high variance).

**distribution shift**

Distribution shift occurs when deployment data differs from training data in language, demographics, seasonality, or product mix. Models degrade silently when shift is unmonitored.

A model trained pre-acquisition fails on the acquired company's ticket vocabulary until retrained or augmented.

#### Worked example

**Situation:** A pilot ML classifier labels incident severity from historical tickets, but performance collapses after a product rename changes customer vocabulary.

**Baseline:** Memorize exact training phrases with a hash map—perfect on train, useless on deploy.

**Application:** Train/validation split by time, fit models of increasing capacity (linear → bigram → small neural), plot train vs validation error, and document distribution shift when product codenames change.

**Test cases:** (1) Normal: phrasing seen in training month. (2) Boundary: new product name with same failure semantics. (3) Adversarial: label noise—mis-tagged P3 tickets marked P1.

**Measurement:** Pre/post-rename F1, learning curves, and slice error on renamed-product tickets.

**Design question:** What evidence distinguishes overfitting from distribution shift on the rename slice?

**Predict before running.** A system is useful when it performs under future conditions, not merely on its training examples. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 1.5: Learning and Generalization"""

data = [(1, 0), (2, 0), (3, 1), (4, 1), (5, 2)]
labels = [0, 0, 1, 1, 0]
def mse_line(m, b, pts):
    return sum((m*x + b - y)**2 for x, y in pts) / len(pts)
best = min(((m, b, mse_line(m, b, data)) for m in [0, 0.5, 1.0] for b in [0, 0.5]), key=lambda t: t[2])
holdout = [(6, 1), (7, 1)]
test_err = mse_line(best[0], best[1], holdout)
print({"fit": best[:2], "train_mse": round(best[2], 3), "holdout_mse": round(test_err, 3)})


#### Interpret and apply

Fit increasingly flexible models to a small noisy dataset and plot errors.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. What observable pattern indicates memorization rather than generalization on tickets?
2. How does a time-based split change your interpretation of validation error?
3. What is the minimal model family for a severity baseline?

<details><summary>Check your reasoning</summary>

Q1: Zero training error with high validation error on paraphrased tickets. Q2: Random split hides temporal shift; time split surfaces rename failures. Q3: Linear or logistic model on bag-of-words.

</details>


### Lecture 06 · Engineering with Uncertainty

Translate uncertain predictions into decisions with thresholds, costs, calibration, fallback behavior, and human oversight.

#### When an uncertain alert still needs action

The estimator assigns a 0.4 probability that a lift report is urgent. Manual review costs about two minutes. Leaving a rider stranded much longer would be far more costly than a review.

A probability estimate can inform two policies with different action costs. Route midrange scores to a reviewer and choose the thresholds using measured errors and staffing capacity. Check that 0.4 means something similar across the stations before carrying it into a new policy.

#### Read the mechanism

**calibration**

Calibration means predicted probabilities align with observed frequencies—70% confidence should be right about 70% of the time. Uncalibrated scores mislead threshold and cost decisions.

A medical triage model with miscalibrated probabilities causes undertriage when 0.9 confidence actually means 0.6 accuracy.

**decision thresholds**

Decision thresholds turn continuous scores into actions—approve, escalate, or abstain. They encode business costs and should be tuned on validation data, not defaults.

Raising a fraud threshold reduces false positives but increases missed fraud; the optimum depends on chargeback cost.

**expected cost**

Expected cost combines probabilities of outcomes with their business costs to rank decisions. It makes asymmetric errors explicit instead of hiding them in accuracy.

Approving a loan when P(default)=0.08 is cheap only if the expected loss is below the interest margin.

**abstention**

Abstention lets a system refuse or defer when confidence is insufficient, routing cases to humans or safer paths. It prevents forced wrong answers on ambiguous inputs.

A benefits bot abstains on incomplete forms instead of guessing eligibility that triggers appeals.

**human review**

Human review inserts expert judgment for high-impact or low-confidence decisions. Designing the queue—what gets reviewed, SLA, feedback loop—determines ROI.

Loan officers review only applications where the model score falls in the 0.4–0.6 band, covering 12% of volume at 3× higher fraud catch.

#### Worked example

**Situation:** The classifier outputs P1 probability 0.72; policy must decide whether to page on-call given asymmetric costs of false alarms vs missed outages.

**Baseline:** Always page when probability > 0.5 regardless of cost or calibration.

**Application:** Plot reliability diagram, pick threshold minimizing expected cost (false page $200 vs missed outage $50k), add abstention band sending borderline tickets to human triage.

**Test cases:** (1) Normal: calibrated 0.9 on confirmed outage. (2) Boundary: 0.55 after Platt scaling on small val set. (3) Adversarial: model overconfident on marketing "urgency" language.

**Measurement:** Expected cost curve vs threshold, ECE calibration error, and page rate at chosen policy.

**Design question:** Why should threshold selection happen in a decision layer separate from the scoring model?

**Predict before running.** Prediction and decision are separate layers; consequences belong in the decision layer. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 1.6: Engineering with Uncertainty"""

scores = [0.92, 0.61, 0.48, 0.33]
COST = {"fp": 200, "fn": 50000}
def expected_cost(threshold):
    decisions = [s >= threshold for s in scores]
    truth = [True, True, False, False]
    fp = sum(d and not t for d, t in zip(decisions, truth))
    fn = sum(not d and t for d, t in zip(decisions, truth))
    return fp * COST["fp"] + fn * COST["fn"]
for t in [0.5, 0.6, 0.7, 0.8]:
    print(f"threshold={t} expected_cost={expected_cost(t)}")


#### Interpret and apply

Design a decision policy for a high-cost false-positive scenario.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. What goes wrong if you tune decision thresholds on the training set?
2. How would miscalibration appear in an incident paging policy?
3. What baseline policy ignores model scores entirely?

<details><summary>Check your reasoning</summary>

Q1: Threshold overfits noise, inflates false pages on holdout months. Q2: High scores on non-outages in reliability diagram. Q3: Always-page or keyword-only paging with fixed rules.

</details>


**Section checkpoint.** Assign separate costs to a wrong route and a human review. Specify when review becomes cheaper.


## Course workshop

Cover the route with a blocked cell and compare the path returned by each search. Record when the heuristic saves work.

Run the reference implementation, then run the checks below it.


In [ ]:
"""Compare breadth-first search and A* on the same grid."""
from collections import deque
from heapq import heappop, heappush

GRID = ["S...#", ".##..", "...#.", ".#...", "...G."]
START, GOAL = (0, 0), (4, 3)


def neighbors(node):
    r, c = node
    for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(GRID) and 0 <= nc < len(GRID[0]) and GRID[nr][nc] != "#":
            yield nr, nc


def reconstruct(parent, node):
    path = [node]
    while node in parent:
        node = parent[node]
        path.append(node)
    return list(reversed(path))


def bfs():
    queue, parent, seen = deque([START]), {}, {START}
    expanded = 0
    while queue:
        node = queue.popleft(); expanded += 1
        if node == GOAL:
            return reconstruct(parent, node), expanded
        for nxt in neighbors(node):
            if nxt not in seen:
                seen.add(nxt); parent[nxt] = node; queue.append(nxt)


def astar():
    frontier, parent, cost = [(0, START)], {}, {START: 0}
    expanded = 0
    while frontier:
        _, node = heappop(frontier); expanded += 1
        if node == GOAL:
            return reconstruct(parent, node), expanded
        for nxt in neighbors(node):
            new_cost = cost[node] + 1
            if nxt not in cost or new_cost < cost[nxt]:
                cost[nxt] = new_cost; parent[nxt] = node
                heuristic = abs(nxt[0] - GOAL[0]) + abs(nxt[1] - GOAL[1])
                heappush(frontier, (new_cost + heuristic, nxt))


if __name__ == "__main__":
    for name, search in (("BFS", bfs), ("A*", astar)):
        path, expanded = search()
        print(f"{name}: path_length={len(path)-1}, expanded={expanded}, path={path}")


### Verify and extend

Run the assertions. Then change one assumption and add a check for the predicted behavior.


In [ ]:
bfs_path, bfs_expanded = bfs()
astar_path, astar_expanded = astar()
assert len(bfs_path) == len(astar_path)
assert astar_expanded <= bfs_expanded
print('Workshop checks passed')


## Course project

Build and compare a rule-based solver, a search-based solver, and a learned predictor for one bounded problem.

Keep your baseline, results, and one failure case so someone else can review the decision you made.
